In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

# DataLoader for training data
import torch
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get the first fold's split
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    break # Use only the first split

# 1. Convert to Tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) # Add a dimension for binary classification
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print("Tensor Shapes:")
print(f"X_train_tensor: {X_train_tensor.shape}")
print(f"y_train_tensor: {y_train_tensor.shape}")
print(f"X_test_tensor: {X_test_tensor.shape}")
print(f"y_test_tensor: {y_test_tensor.shape}")




In [ ]:
# 2. Create TensorDataset objects
# Create Datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print("\nNumber of samples in datasets:")
print(f"Train dataset: {len(train_dataset)}")
print(f"Test dataset: {len(test_dataset)}")



In [ ]:
# 3. Create DataLoaders

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# 4. Print shape of one batch
# 4. Inspect Data: Print the shape of one batch from the train loader
print("\nInspecting one batch from train_loader:")
for features_batch, labels_batch in train_loader:
    print(f"Features Batch Shape: {features_batch.shape}")
    print(f"Labels Batch Shape: {labels_batch.shape}")


In [ ]:
# 5. Display sample images
    # 5. Display a few samples (adapting 'display images' for tabular data)
    print("\nFirst 5 samples from the batch:")
    print("Features:\n", features_batch[:5])
    print("Labels:\n", labels_batch[:5])
    break # Only inspect the first batch


In [ ]:
# Task 1: Write your model class here:
from tqdm.auto import tqdm

def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, labels) in enumerate(tqdm(data_loader, desc="Training")):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    return avg_loss

print("train_epoch function defined successfully.")

In [ ]:
# Task 2: Write your training loop here:


In [ ]:
# Task 2: Write your training loop here:
from tqdm.auto import tqdm

def validate_epoch(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(tqdm(data_loader, desc="Validation")):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    return avg_loss

print("validate_epoch function defined successfully.")

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters for dummy data and training
INPUT_SIZE = 10
OUTPUT_SIZE = 1
NUM_EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# 2. Model Initialization
model = SimpleNN(input_size=INPUT_SIZE, output_size=OUTPUT_SIZE).to(device)
print(f"Model initialized and moved to {device}.")

# 3. Loss Function
loss_fn = nn.MSELoss()
print("Loss function (MSELoss) defined.")

# 4. Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Optimizer (Adam) defined.")

# 6. Data Generation (Dummy Data)
# Generate random dummy data for training
X_train = torch.randn(1000, INPUT_SIZE)
y_train = torch.randn(1000, OUTPUT_SIZE)

# Generate random dummy data for validation
X_val = torch.randn(200, INPUT_SIZE)
y_val = torch.randn(200, OUTPUT_SIZE)

# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("Dummy training and validation data loaders created.")

# 7. Training Loop
train_losses = []
val_losses = []

print("\nStarting main training and validation loop...")
for epoch in range(NUM_EPOCHS):
    avg_train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device)
    avg_val_loss = validate_epoch(model, val_loader, loss_fn, device)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

# 8. Summarize Results
final_train_loss = train_losses[-1]
final_val_loss = val_losses[-1]

print("\nTraining and validation complete!")
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")

In [ ]:
# Task 3: Write your validation loop here:

In [ ]:
# Task 4: Define device, model, loss, optimizer:

In [ ]:
# Task 5: Start training for 20 epochs:

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: